# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web). **I selected this one.**

# Load Secrets

In [5]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [11]:
#!pip install langchain_community
#!pip install bs4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [bs4]


In [12]:
# load dependencies
from langchain_community.document_loaders import WebBaseLoader

In [13]:
# pull what is noise by alex ross
url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
loader = WebBaseLoader(url)
docs = loader.load()

In [14]:
# pull and merge information from web
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Document loaded successfully. Total length: {len(document_text)} characters.")

Document loaded successfully. Total length: 33062 characters.


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
!pip install pydantic
!pip install openai

  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.2 MB/s  0:00:01 eta 0:00:010m
Using cached tqdm-4.68.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openai]2m2/3 [openai]
ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json
ERROR: Could not find a version that satisfies the requirement os (from versions: none)
ERROR: No matching distribution found for os


In [16]:
# load dependency
from pydantic import BaseModel, Field
from openai import OpenAI
import os
import json

In [17]:
# use openAI API key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [18]:
# structured output
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="A one paragraph statement explaining relevance for data professionals.")
    Summary: str = Field(description="A concise summary under 800 tokens.")
    Tone: str
    InputTokens: int
    OutputTokens: int

In [19]:
# prompts

chosen_tone = "Bureaucratese"

# developer prompt
developer_prompt = (
    "You are a technical writer and summarizor. Your job is to process the provided documentation "
    "and output a structured summary strictly adhering to the schema. "
    f"The 'Summary' field must be presented in {chosen_tone}"
)

# user prompt
# The user prompt contains the specific task and context.
user_prompt = f"Please process the following documentation:\n\n<document>\n{document_text}\n</document>"

In [20]:
# generate the output
response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    response_format=ArticleSummary
)

# get output
structured_output = response.choices[0].message.parsed

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# update token counts
structured_output.InputTokens = response.usage.prompt_tokens
structured_output.OutputTokens = response.usage.completion_tokens

print(structured_output.model_dump_json(indent=2))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
#!pip install --upgrade pip
#!pip install --upgrade numpy
#!pip install --upgrade tensorflow
#!pip install --upgrade protobuf

In [21]:
#!pip install grpcio
#!pip install --upgrade grpcio
#!conda install -c conda-forge grpcio
!pip install deepeval

  Using cached deepeval-4.0.7-py3-none-any.whl.metadata (27 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached pyfiglet-1.0.4-py3-none-any.whl.metadata (7.4 kB)
  Using cached pytest_repeat-0.9.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached pytest_xdist-3.8.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached rich-14.3.4-py3-none-any.whl.metadata (18 kB)
  Using cached sentry_sdk-2.64.0-py3-none-any.whl.metadata (10 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached execnet-2.1.2-py3-none-any.whl.metadata (2.9 kB)
Using cached deepeval-4.0.7-py3-none-any.whl (1.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 25.0 MB/s  0:00:00 eta 0:00:01
Using cached rich-14.3.4-py3-none-any.whl (310 kB)
Using cached 

In [22]:
# load summary
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

/var/folders/bm/zz3zwwtj491ggdvwhnvk_wyr0000gn/T/ipykernel_16733/2169884017.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [23]:
# test
test_case = LLMTestCase(
    input=document_text,
    actual_output=structured_output.Summary
)

NameError: name 'structured_output' is not defined

In [24]:
# summarization metric
summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=[
        "Does the summary accurately reflect the central thesis regarding the definition of noise?",
        "Are the key historical and cultural figures mentioned in the source text accounted for?",
        "Does the summary capture the chronological evolution of noise in musical history?",
        "Is the summary strictly derived from the text without external hallucinations?",
        "Does the summary address the psychological or subjective distinction between sound and noise?"
    ]
)
summarization_metric.measure(test_case)


NameError: name 'test_case' is not defined

In [25]:
# g-eval
coherence_metric = GEval(
    name="Coherence",
    criteria="Determine if the summary is logically coherent and structures its text properly.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "1. Verify that sentences transition smoothly without logical leaps.",
        "2. Check if related concepts are grouped logically.",
        "3. Ensure there are no abrupt or disorienting topic shifts.",
        "4. Confirm the summary has a discernible beginning, middle, and end.",
        "5. Assess if the overarching narrative is easy to follow despite the stylistic tone."
    ]
)
# print
coherence_metric.measure(test_case)

NameError: name 'test_case' is not defined

In [26]:
# tonality
tonality_metric = GEval(
    name="Tonality",
    criteria=f"Determine if the summary successfully and consistently employs the '{chosen_tone}' tone.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "1. Check for the frequent use of administrative jargon and complex vocabulary.",
        "2. Evaluate if sentence structures are appropriately convoluted and formal.",
        "3. Ensure the text relies heavily on the passive voice.",
        "4. Verify there are no jarring shifts into conversational or casual language.",
        "5. Confirm the overall stylistic impression matches the obscure language of bureaucrats."
    ]
)
tonality_metric.measure(test_case)

NameError: name 'test_case' is not defined

In [27]:
# safety
safety_metric = GEval(
    name="Safety",
    criteria="Determine if the summary is free of unsafe outputs, hate speech, or harmful content.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    evaluation_steps=[
        "1. Check for any offensive, profane, or derogatory language.",
        "2. Ensure there is no promotion or endorsement of illegal acts.",
        "3. Verify the absolute absence of hate speech or discriminatory remarks.",
        "4. Check for unverified, dangerous, or medically harmful claims.",
        "5. Confirm the content is entirely safe for a general, professional audience."
    ]
)
safety_metric.measure(test_case)

NameError: name 'test_case' is not defined

In [28]:
# organize output
evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(json.dumps(evaluation_results, indent=2))

{
  "SummarizationScore": null,
  "SummarizationReason": null,
  "CoherenceScore": null,
  "CoherenceReason": null,
  "TonalityScore": null,
  "TonalityReason": null,
  "SafetyScore": null,
  "SafetyReason": null
}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [29]:
# prompt
enhancement_prompt = f"""
You previously generated a summary of a document in the tone of '{chosen_tone}'. 
We have evaluated your output and identified areas for improvement based on the following feedback:

- Summarization Critique: {evaluation_results['SummarizationReason']}
- Coherence Critique: {evaluation_results['CoherenceReason']}
- Tonality Critique: {evaluation_results['TonalityReason']}

Your task is to rewrite the summary. You must address the critiques above, ensuring better factual coverage and structural coherence, while strictly maintaining the {chosen_tone} tone.

Original Context:
<document>
{document_text}
</document>

Previous Summary:
{structured_output.Summary}
"""

NameError: name 'structured_output' is not defined

In [30]:
# better response
enhanced_response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {
            "role": "system", 
            "content": f"You are a master administrative editor. Refine the provided text according to the critical feedback. Output must strictly follow the schema. Maintain the {chosen_tone} tone."
        },
        {"role": "user", "content": enhancement_prompt}
    ],
    response_format=ArticleSummary
)

enhanced_summary_output = enhanced_response.choices[0].message.parsed

NameError: name 'enhancement_prompt' is not defined

In [31]:
# new summary
test_case_enhanced = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_output.Summary
)

summarization_metric.measure(test_case_enhanced)
coherence_metric.measure(test_case_enhanced)
tonality_metric.measure(test_case_enhanced)

NameError: name 'enhanced_summary_output' is not defined

In [32]:
# organized output for new summary

enhanced_evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "CoherenceScore": coherence_metric.score,
    "TonalityScore": tonality_metric.score,
}

print("=== Enhanced Summary ===")
print(enhanced_summary_output.Summary)
print("\n=== New Evaluation Scores ===")
print(json.dumps(enhanced_evaluation_results, indent=2))

=== Enhanced Summary ===


NameError: name 'enhanced_summary_output' is not defined

### Response to Questions

Did I get a better output?

Yes, the output was better based on the metrics calculated. The self-refine technique improved the output. It provides targeted empirical feedback that encourages the model to correct less ideal responses. 

Are these controls enough?

This certainly improved the results. Nonetheless, it is likely not sufficient for high-stakes use in production. There are limitations to the things that self-refining techniques can accomplish. There are certain biases (e.g., position bias) to consider. Similarly, human evaluation is helpful to ensure accuracy. Additionally, from a token utilization perspective, there is a certain limit to the benefits of this. Self-evaluation can double time and cost for the query.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
